# 05. Consolidación de outliers, evolución temporal y proyecciones 2D



Toma el modelo calibrado en el paso anterior, agrupa los documentos residuales (`-1`) bajo el **Tópico 100** (misceláneo), reordena los tópicos temáticos por volumen de artículos y genera las series temporales (`topics_over_time`) con detección de picos históricos, los mapas semánticos UMAP 2D y los gráficos exportados para la presentación.

### Entradas
- `outputs/bertopic_acotado/modelo_acotado`
- `outputs/bertopic_acotado/df_modelo_acotado.parquet`
- `outputs/bertopic_acotado/embeddings_naturales.npy`
- `data/processed/stopwords_anfibia.txt`

### Salidas
- `outputs/bertopic_acotado/modelo_completo` (modelo definitivo con Tópico 100 y numeración ordenada por volumen)
- `outputs/bertopic_acotado/df_completo.parquet` (dataset depurado con la columna `topic` final)
- `outputs/bertopic_acotado/graficos/` (mapas UMAP 2D y wordclouds interactivos HTML)
- `outputs/presentacion/` (imágenes PNG y GIF en alta resolución)

---

In [1]:
# EJECUTAR ESTA CELDA PRIMERO QUE NADA (Y REINICIAR EL KERNEL ANTES SI FALLÓ)
import coverage.types as ct

# Parche de compatibilidad para evitar que Numba/UMAP crasheen
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

## 1. Configuración del entorno
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic
import pickle

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
STOPWORDS_PATH = PROJECT_ROOT / "data" / "processed" / "stopwords_anfibia.txt"

assert DATA_PATH.exists(), f"Falta archivo de entrada: {DATA_PATH.name}"
assert STOPWORDS_PATH.exists(), f"Falta archivo de stopwords: {STOPWORDS_PATH.name}"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bertopic_acotado"
GRAFICOS_DIR = OUTPUT_DIR / "graficos"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
GRAFICOS_DIR.mkdir(exist_ok=True, parents=True)

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import copy
from bertopic.backend._utils import select_backend

# 1. Cargar el modelo base
topic_model_original = BERTopic.load(str(OUTPUT_DIR / "modelo_acotado"))

# 2. Copia profunda para no mutar el original en memoria
topic_model = copy.deepcopy(topic_model_original)
raw_embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
topic_model.embedding_model = select_backend(raw_embedding_model)

# 2. Cargar el DataFrame con las asignaciones por nota
df = pd.read_parquet(OUTPUT_DIR / "df_modelo_acotado.parquet")

# 3. Cargar embeddings
embeddings_naturales = np.load(OUTPUT_DIR / "embeddings_naturales.npy")

# # 4. Cargar textos y centroides
# with open(OUTPUT_DIR / "textos_pipeline.pkl", "rb") as f:
#     textos = pickle.load(f)

# with open(OUTPUT_DIR / "variables_intermedias.pkl", "rb") as f:
#     intermedios = pickle.load(f)

2026-11-06 12:55:17,644 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3015.13it/s]


### Ponemos los outliers (-1) en un nuevo topico ID 100 y sacamos sus palabras clave

In [3]:
with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
    stopwords_list = f.read().splitlines()

vectorizer_model = CountVectorizer(stop_words=stopwords_list, min_df=3)
representation_model = KeyBERTInspired(top_n_words=100)
docs_lematizados = df["document_text_final"].fillna("").astype(str).tolist()

# Convertimos el -1 en 100
nuevo_id_outliers = 100
topics_con_100 = [nuevo_id_outliers if t == -1 else t for t in df["topic_umbral"].values]

# Actualizamos el modelo para que el 100 exista formalmente con sus keywords
topic_model.update_topics(
    docs_lematizados,
    topics=topics_con_100,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    top_n_words=100
)

# Sincronizamos en el DataFrame
df["topic"] = topics_con_100

print(f"Outliers convertidos a Tópico {nuevo_id_outliers}.")
print("Keywords asignadas al 100:", [w for w, _ in topic_model.get_topic(nuevo_id_outliers)[:5]])

2026-11-06 12:55:25,681 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers convertidos a Tópico 100.
Keywords asignadas al 100: ['padre', 'hombre', 'pueblo', 'causa', 'humano']


### Reordenamos por cantidad de articulos los tópicos temáticos, sin tocar el 100

In [4]:
# 1. Ranking de volumen SOLO para los tópicos que no son el 100
ranking_tematicos = (
    pd.Series(df["topic"][df["topic"] != nuevo_id_outliers])
    .value_counts()
    .index.tolist()
)

# 2. Mapeo: los temáticos se renumeran 0, 1, 2... y el 100 se queda en 100
mapeo_reorden = {viejo_id: nuevo_id for nuevo_id, viejo_id in enumerate(ranking_tematicos)}
mapeo_reorden[nuevo_id_outliers] = nuevo_id_outliers

# 3. Aplicamos el mapeo final
topics_finales = [mapeo_reorden[t] for t in df["topic"].values]

# 4. Actualizamos el modelo con la numeración definitiva y guardamos en df
topic_model.update_topics(
    docs_lematizados,
    topics=topics_finales,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model
)
df["topic"] = topics_finales

2026-11-06 12:55:39,079 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [5]:
print(f"El ID del nuevo tópico es: {nuevo_id_outliers}")
print("Palabras clave asignadas:")
print(topic_model.get_topic(nuevo_id_outliers))

El ID del nuevo tópico es: 100
Palabras clave asignadas:
[('padre', np.float32(0.48754853)), ('hombre', np.float32(0.4858225)), ('pueblo', np.float32(0.48155522)), ('causa', np.float32(0.46678656)), ('humano', np.float32(0.44766012)), ('género', np.float32(0.44368818)), ('mundo', np.float32(0.44044507)), ('sociedad', np.float32(0.4297413)), ('obra', np.float32(0.42884064)), ('noche', np.float32(0.42653722)), ('libro', np.float32(0.42468467)), ('leer', np.float32(0.411521)), ('cultura', np.float32(0.40966165)), ('cultural', np.float32(0.40862185)), ('década', np.float32(0.4078492)), ('nacional', np.float32(0.40720844)), ('país', np.float32(0.40685707)), ('partido', np.float32(0.40631235)), ('periodista', np.float32(0.40499312)), ('muerto', np.float32(0.40222666)), ('realidad', np.float32(0.39771116)), ('personaje', np.float32(0.3943108)), ('amor', np.float32(0.39411443)), ('morir', np.float32(0.39286587)), ('hijo', np.float32(0.3855177)), ('carlos', np.float32(0.38541508)), ('presidente

In [6]:
info = topic_model.get_topic_info()
display(info[['Topic','Count', 'Name']])

,Topic,Count,Name
0,0,620,0_candidato_electoral_campaña_debate
1,1,392,1_literario_autor_escritor_poeta
2,2,334,2_feminismo_feminista_aborto_abortar
3,3,247,3_muerto_dictadura_muerte_morir
4,4,198,4_ambiental_ambiente_crisis_proyecto
5,5,195,5_déficit_economía_pobreza_económico
6,6,141,6_futbolista_fifa_fútbol_brasil
7,7,126,7_pedagógico_enseñanza_escolar_educativo
8,8,126,8_epidemia_pandemia_vacunar_virus
9,9,119,9_periodístico_mediático_periodista_noticia


In [7]:
# 1. Cantidad de notas y nombre generado
info_outlier = topic_model.get_topic_info(nuevo_id_outliers)
print(f"ID del tópico de outliers: {nuevo_id_outliers}")
print(f"Cantidad de notas: {info_outlier['Count'].values[0]}")
print(f"Palabras clave")

# 2. Top 10 palabras clave con su peso
palabras_clave = topic_model.get_topic(nuevo_id_outliers)
for palabra, peso in palabras_clave:
    print(f"  • {palabra}: {peso:.4f}")

ID del tópico de outliers: 100
Cantidad de notas: 534
Palabras clave
  • padre: 0.4875
  • hombre: 0.4858
  • pueblo: 0.4816
  • causa: 0.4668
  • humano: 0.4477
  • género: 0.4437
  • mundo: 0.4404
  • sociedad: 0.4297
  • obra: 0.4288
  • noche: 0.4265
  • libro: 0.4247
  • leer: 0.4115
  • cultura: 0.4097
  • cultural: 0.4086
  • década: 0.4078
  • nacional: 0.4072
  • país: 0.4069
  • partido: 0.4063
  • periodista: 0.4050
  • muerto: 0.4022
  • realidad: 0.3977
  • personaje: 0.3943
  • amor: 0.3941
  • morir: 0.3929
  • hijo: 0.3855
  • carlos: 0.3854
  • presidente: 0.3785
  • martín: 0.3759
  • derecho: 0.3746
  • hermano: 0.3706
  • muerte: 0.3644
  • chico: 0.3644
  • niño: 0.3638
  • familiar: 0.3581
  • futuro: 0.3564
  • subir: 0.3545
  • san: 0.3460
  • compañero: 0.3429
  • mujer: 0.3421
  • voz: 0.3397
  • aprender: 0.3388
  • cambio: 0.3347
  • colectivo: 0.3344
  • identidad: 0.3326
  • hija: 0.3323
  • investigación: 0.3296
  • música: 0.3296
  • movimiento: 0.3271
 

In [8]:
topic_model.visualize_barchart(top_n_topics=50, n_words=8)

### Topicos en el tiempo: Evolucion, articulos clave, etc.

In [9]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# ==============================================================================
# 1. CÁLCULO DE EVOLUCIÓN TEMPORAL
# ==============================================================================
print("Calculando evolución temporal...")

# Asegurar formato de fechas
df["post_date_dt"] = pd.to_datetime(df["post_date"].str.split(" ").str[0], format="%Y-%m-%d", errors="coerce")

# 1. Filtramos registros que tengan fecha válida y texto
mascara_validos = (
    df["post_date_dt"].notna() 
    & df["document_text_final"].fillna("").astype(str).str.strip().ne("")
).to_numpy()

docs_filtrados = df.loc[mascara_validos, "document_text_final"].astype(str).tolist()
fechas_filtradas = df.loc[mascara_validos, "post_date_dt"].dt.to_pydatetime().tolist()

# 2. Pasamos EXACTAMENTE las asignaciones de esos documentos
topics_filtrados = np.array(topics_finales)[mascara_validos].tolist()

# 3. Forzamos a topics_over_time a usar los tópicos ya asignados sin recalcular
topics_over_time = topic_model.topics_over_time(
    docs=docs_filtrados,
    topics=topics_filtrados,
    timestamps=fechas_filtradas,
    nr_bins=40
)

# Sincronizamos las asignaciones finales en el DataFrame
top_topics_ids = [t for t in topic_model.get_topic_info()["Topic"].tolist() if t != -1]

# Diccionario de nombres limpios
topic_names_dict = {}
for t_id in top_topics_ids:
    palabras = topic_model.get_topic(t_id)
    if palabras:
        topic_names_dict[t_id] = f"{t_id}_" + "_".join([w for w, _ in palabras[:3]])
    else:
        topic_names_dict[t_id] = f"Topic_{t_id}"

Calculando evolución temporal...


In [10]:
# ==============================================================================
# 2. DETECCIÓN DE PICOS Y NOTAS CLAVE
# ==============================================================================
TOP_N_NOTAS_PICO = 3
TOP_N_NOTAS_NORMAL = 1
MIN_FREQ_PICO = 3

unique_timestamps = sorted(pd.to_datetime(topics_over_time["Timestamp"].unique()))
timeline_data = []

print("Extrayendo picos y artículos representativos...")

for t_id in top_topics_ids:
    df_t = topics_over_time[topics_over_time["Topic"] == t_id].sort_values("Timestamp").reset_index(drop=True)
    if df_t.empty:
        continue
    
    t_name = topic_names_dict.get(t_id, f"Topic_{t_id}")
    freqs = df_t["Frequency"].values
    umbral = max(MIN_FREQ_PICO, freqs.mean())
    
    if len(freqs) > 2:
        mayor_al_anterior = np.r_[False, freqs[1:] > freqs[:-1]]
        mayor_al_siguiente = np.r_[freqs[:-1] > freqs[1:], False]
        picos_locales = mayor_al_anterior & mayor_al_siguiente & (freqs >= umbral)
        picos_idx = set(np.where(picos_locales)[0])
    else:
        picos_idx = set()
        
    if len(freqs) > 0:
        picos_idx.add(np.argmax(freqs))
    
    for i, row in df_t.iterrows():
        fecha_dt = pd.to_datetime(row["Timestamp"])
        freq = int(row["Frequency"])
        is_peak = (i in picos_idx)
        is_max = (i == np.argmax(freqs))
        
        idx_bin = unique_timestamps.index(fecha_dt)
        t_start = unique_timestamps[idx_bin - 1] if idx_bin > 0 else fecha_dt - pd.DateOffset(months=3)
        t_end = unique_timestamps[idx_bin + 1] if idx_bin < len(unique_timestamps) - 1 else fecha_dt + pd.DateOffset(months=3)
        
        mask_bin = (df["topic"] == t_id) & (df["post_date_dt"] >= t_start) & (df["post_date_dt"] <= t_end)
        indices_bin = df[mask_bin].index.tolist()
        
        n_necesarias = TOP_N_NOTAS_PICO if is_peak else TOP_N_NOTAS_NORMAL
        if len(indices_bin) < n_necesarias:
            mask_año = (df["topic"] == t_id) & (df["post_date_dt"].dt.year == fecha_dt.year)
            indices_bin = df[mask_año].index.tolist()
            
        notas_list = []
        if indices_bin:
            ref_vector = embeddings_naturales[indices_bin].mean(axis=0, keepdims=True)
            similitudes = cosine_similarity(embeddings_naturales[indices_bin], ref_vector).flatten()
            top_locales = np.argsort(similitudes)[::-1][:n_necesarias]
            
            for rank, idx_local in enumerate(top_locales, start=1):
                idx_global = indices_bin[idx_local]
                nota = df.loc[idx_global]
                notas_list.append({
                    "Titulo": nota.get("title_text", "Sin título"),
                    "Tagline": str(nota.get("tagline", ""))[:80] if pd.notna(nota.get("tagline")) else "",
                    "Tag": str(nota.get("post_tag", "")).split(";")[0].strip() if pd.notna(nota.get("post_tag")) else ""
                })
                
        timeline_data.append({
            "Topic": t_id,
            "Name": t_name,
            "Fecha": fecha_dt,
            "Frecuencia": freq,
            "Es_Pico": is_peak,
            "Es_Maximo": is_max,
            "Articulos": notas_list
        })

Extrayendo picos y artículos representativos...


In [11]:
# ==============================================================================
# 3. RENDERIZADO DEL GRÁFICO PLOTLY
# ==============================================================================
TOP_N_TRAZAS_ACTIVAS = 3
timeline_dict = {(row["Topic"], row["Fecha"]): row for row in timeline_data}

fig = topic_model.visualize_topics_over_time(topics_over_time, topics=top_topics_ids)
colors = px.colors.qualitative.G10
rombos_picos = []

for i, t in enumerate(fig.data):
    c = colors[i % len(colors)]
    vis = True if i < TOP_N_TRAZAS_ACTIVAS else 'legendonly'
    
    raw_name = t.name
    try:
        topic_id = int(raw_name.split("_")[0])
    except (ValueError, IndexError):
        continue

    clean_name = raw_name.replace("_", " ")
    textos_hover_linea = []
    
    for x_val, y_val in zip(t.x, t.y):
        fecha_dt = pd.to_datetime(x_val)
        info_punto = timeline_dict.get((topic_id, fecha_dt))
        
        if info_punto and info_punto["Articulos"]:
            if info_punto["Es_Pico"]:
                detalles_hover = ""
                for n in info_punto["Articulos"]:
                    titulo = n["Titulo"][:39] + ("..." if len(n["Titulo"]) > 42 else "")
                    tagline_str = f"<br>&nbsp;&nbsp;&nbsp;&nbsp;<i>\"{n['Tagline'][:72]}...\"</i>" if n["Tagline"] and n["Tagline"].lower() != "nan" else ""
                    tag_str = f" <b>(#{n['Tag']})</b>" if n["Tag"] and n["Tag"].lower() != "nan" else ""
                    detalles_hover += f"<br>• <b>{titulo}</b>{tag_str}{tagline_str}"
                
                label_pico = "Máximo Histórico" if info_punto["Es_Maximo"] else "Pico Destacado"
                textos_hover_linea.append(
                    f"<br><b>[{label_pico}]</b><br><b>Artículos representativos:</b>{detalles_hover}"
                )
            else:
                tit = info_punto["Articulos"][0]["Titulo"]
                tit_corto = (tit[:45] + "...") if len(tit) > 48 else tit
                textos_hover_linea.append(f"<br>• <b>Nota del mes:</b> {tit_corto}")
        else:
            textos_hover_linea.append("")

        if info_punto and info_punto["Es_Maximo"]:
            detalles_rombo = ""
            for n in info_punto["Articulos"]:
                titulo = n["Titulo"][:39] + ("..." if len(n["Titulo"]) > 42 else "")
                tagline_str = f"<br>&nbsp;&nbsp;&nbsp;&nbsp;<i>\"{n['Tagline'][:72]}...\"</i>" if n["Tagline"] and n["Tagline"].lower() != "nan" else ""
                tag_str = f" <b>(#{n['Tag']})</b>" if n["Tag"] and n["Tag"].lower() != "nan" else ""
                detalles_rombo += f"<br>• <b>{titulo}</b>{tag_str}{tagline_str}"
                
            if not info_punto["Articulos"]:
                detalles_rombo = "<br>• Sin notas exactas en el período"

            rombos_picos.append(
                go.Scatter(
                    x=[fecha_dt], 
                    y=[info_punto["Frecuencia"]], 
                    mode="markers", 
                    showlegend=False, 
                    visible=vis, 
                    legendgroup=f"g{topic_id}",
                    marker=dict(size=12, color=c, symbol="diamond", line=dict(width=1.5, color="#111")),
                    hovertemplate=(
                        f"<b>Máximo Histórico: {clean_name}</b><br>"
                        "Fecha: %{x|%Y-%m}<br>"
                        "Frecuencia: %{y}<br>"
                        f"<b>Artículos representativos:</b>{detalles_rombo}"
                        "<extra></extra>"
                    )
                )
            )

    t.update(
        name=clean_name, 
        visible=vis, 
        legendgroup=f"g{topic_id}", 
        line=dict(color=c, width=2.5),
        customdata=textos_hover_linea,
        hovertemplate=(
            f"<b>{clean_name}</b><br>"
            "Fecha: %{x|%Y-%m}<br>"
            "Frecuencia: %{y}"
            "%{customdata}<extra></extra>"
        )
    )

fig.add_traces(rombos_picos)

fig.update_layout(
    title="<b>Tópicos a través del tiempo (con Outliers integrados)</b>", 
    plot_bgcolor='rgba(0,0,0,0)', 
    paper_bgcolor='#fff', 
    height=750, 
    hovermode="closest",
    hoverlabel=dict(bgcolor="white", font_size=16)
)
fig.update_yaxes(showgrid=True, gridcolor="#ebebeb")
fig.update_xaxes(showgrid=True, gridcolor="#ebebeb")

In [12]:
# 1. Filtrar los artículos asignados al tópico 20
notas_t20 = df[df["topic"] == 21][[
    "ID", "post_date", "title_text", "tagline", "post_tag"
]].sort_values("post_date")

print(f"Total de notas del Tópico 20: {len(notas_t20)}")
print("Rango de fechas:", notas_t20["post_date"].min(), "a", notas_t20["post_date"].max())

# 2. Ver títulos, bajadas y etiquetas
display(notas_t20)

# 3. Top palabras clave según el modelo
print("\nPalabras clave:")
for palabra, score in topic_model.get_topic(21):
    print(f"{palabra}: {score:.4f}")

print("\nPosts 'Día del Padre': ", df["post_tag"].fillna("").str.contains("Día del Padre").sum())

Total de notas del Tópico 20: 12
Rango de fechas: 2015-06-20 07:40:54 a 2015-06-20 08:09:33


,ID,post_date,title_text,tagline,post_tag
429,25403,2015-06-20 07:40:54,La habitación 125 es una nave espacial,NaN,Día del Padre
431,25409,2015-06-20 07:46:02,Un monstruoso colibrí,NaN,Día del Padre
432,25411,2015-06-20 07:47:21,Susurros en el agua,NaN,Día del Padre
435,25417,2015-06-20 07:51:21,Un genio de la filosofía,NaN,Día del Padre
436,25419,2015-06-20 07:52:41,Hoy fue un día hermoso en las Malvinas,NaN,Día del Padre
437,25421,2015-06-20 07:54:00,Con vos en el fitito azul,NaN,Día del Padre
442,25432,2015-06-20 08:00:45,Papá siempre fue un barrilete,NaN,Día del Padre
443,25434,2015-06-20 08:01:31,El golpe de Papá,NaN,Día del Padre
447,25443,2015-06-20 08:05:47,Mi papá se levanta de noche,NaN,Día del Padre
448,25447,2015-06-20 08:07:25,Viaje a Rosario de la Frontera,NaN,Día del Padre



Palabras clave:
papá: 0.6692
padre: 0.6289
abuelo: 0.5818
hermano: 0.4603
hombre: 0.3882
mamá: 0.3749
filosofar: 0.3512
filosofía: 0.3373
abuela: 0.3230
subir: 0.3166
estár: 0.3072
despertar: 0.3056
genio: 0.3010
sentar: 0.2945
acabar: 0.2915
querés: 0.2878
timidez: 0.2876
recuerdo: 0.2802
gata: 0.2786
pelo: 0.2781
cabeza: 0.2753
jardín: 0.2751
levantar: 0.2730
nan: 0.2726
visitar: 0.2696
leo: 0.2664
arrancar: 0.2663
traer: 0.2659
angustia: 0.2637
fondo: 0.2623
silencio: 0.2579
noche: 0.2576
árbol: 0.2562
choza: 0.2561
pila: 0.2518
casa: 0.2516
agarrar: 0.2488
morir: 0.2449
diente: 0.2427
mundo: 0.2426
ey: 0.2401
carta: 0.2367
ginkgo: 0.2359
mueble: 0.2311
palo: 0.2304
golpe: 0.2273
mirada: 0.2271
zapiola: 0.2254
fogón: 0.2249
plantar: 0.2223
alejar: 0.2199
lejos: 0.2180
caparazón: 0.2172
servilleta: 0.2170
bollo: 0.2144
rosario: 0.2033
colibrí: 0.2000
ala: 0.1966
atrás: 0.1927
embelesado: 0.1871
volar: 0.1815
pelu: 0.1814
balde: 0.1789
muerte: 0.1769
gregorio: 0.1764
atropellar: 0.17

### Articulos mas importantes de cada topico

In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

TOP_N_NOTAS = 3         # Cantidad de artículos por tópico
TOP_N_TOPICOS = None    # None para extraer de todos (o un entero ej: 10)

topicos_info = topic_model.get_topic_info()
filas_representativas = []

for _, row in topicos_info.iterrows():
    t_id = int(row["Topic"])
    t_name = row["Name"]
    
    # Índices de las notas asignadas a este tópico
    indices_topico = df[df["topic"] == t_id].index.tolist()
    
    if not indices_topico:
        continue
        
    # 1. Centroide semántico del tópico en base a los embeddings naturales
    centroide = embeddings_naturales[indices_topico].mean(axis=0, keepdims=True)
    
    # 2. Similitud coseno de cada artículo respecto al centroide
    similitudes = cosine_similarity(embeddings_naturales[indices_topico], centroide).flatten()
    
    # 3. Top N artículos más cercanos al núcleo temático
    top_locales = np.argsort(similitudes)[::-1][:TOP_N_NOTAS]
    
    for rank, idx_local in enumerate(top_locales, start=1):
        idx_global = indices_topico[idx_local]
        nota = df.loc[idx_global]
        
        filas_representativas.append({
            "Topic": t_id,
            "Topic_Name": t_name,
            "Rank": rank,
            "Similitud_Centroide": round(float(similitudes[idx_local]), 4),
            "ID": nota.get("ID", nota.get("doc_id", idx_global)),
            "Titulo": nota.get("title_text", "Sin título"),
            "URL": str(nota.get("source_url", "")).strip(),
            "Fecha": str(nota.get("post_date", ""))[:10],
            "Tagline": str(nota.get("tagline", ""))[:90] if pd.notna(nota.get("tagline")) else "",
            "Tag": str(nota.get("post_tag", "")).split(";")[0].strip() if pd.notna(nota.get("post_tag")) else ""
        })

df_representativos = pd.DataFrame(filas_representativas)

# Vista previa
if TOP_N_TOPICOS is not None:
    topicos_unicos = df_representativos["Topic"].unique()[:TOP_N_TOPICOS]
    display(df_representativos[df_representativos["Topic"].isin(topicos_unicos)])
else:
    display(df_representativos.head(20))

,Topic,Topic_Name,Rank,Similitud_Centroide,ID,Titulo,URL,Fecha,Tagline,Tag
0,0,0_candidato_electoral_campaña_debate,1,0.8916,141535,El desafío de gobernar una sociedad rota,https://www.revistaanfibia.com/el-desafio-de-g...,2019-10-30,El regreso del peronismo,
1,0,0_candidato_electoral_campaña_debate,2,0.8842,191049,Dios atiende en Buenos Aires,https://www.revistaanfibia.com/dios-atiende-en...,2023-10-26,La política nacional se irradia desde el AMBA,Elecciones 2023
2,0,0_candidato_electoral_campaña_debate,3,0.8781,161639,Lecciones de las elecciones,https://www.revistaanfibia.com/lecciones-de-la...,2021-11-18,De las Legislativas al 2023,Elecciones
3,1,1_literario_autor_escritor_poeta,1,0.8285,68548,Las otras infancias clandestinas,https://www.revistaanfibia.com/las-otras-infan...,2017-05-24,Nuevas voces de la memoria,24M: Día de la memoria por la verdad y la just...
4,1,1_literario_autor_escritor_poeta,2,0.8230,56461,Universo Lestido,https://www.revistaanfibia.com/universo-lestido/,2017-01-05,Fotografía e intimidad,
5,1,1_literario_autor_escritor_poeta,3,0.8181,25107,Todos anfibios,https://www.revistaanfibia.com/todos-anfibios/,2015-06-11,Al rescate,
6,2,2_feminismo_feminista_aborto_abortar,1,0.8824,153752,Las madres sin cuota salen del closet,https://www.revistaanfibia.com/cuota-alimentar...,2021-06-25,Paternidad morosa: ¿por qué no paga?,
7,2,2_feminismo_feminista_aborto_abortar,2,0.8605,64461,"Ni machos, ni fachos",https://www.revistaanfibia.com/ni-machos-ni-fa...,2017-04-12,Una agenda feminista para la Justicia,#NiUnaMenos
8,2,2_feminismo_feminista_aborto_abortar,3,0.8492,153102,Desarmar la arquitectura neoliberal,https://www.revistaanfibia.com/megaelecciones-...,2021-05-18,Megaelecciones en Chile: conversaciones femini...,Chile
9,3,3_muerto_dictadura_muerte_morir,1,0.8410,94944,Se ahogó y punto,https://www.revistaanfibia.com/se-ahogo-punto/,2018-02-01,Santiago Maldonado,


### Proyecciones UMAP 2D y nubes de palabras por tópico

Proyecta los embeddings de documentos y los centroides de tópicos a 2D (UMAP con métrica coseno) destacando el artículo más representativo de cada clúster, y genera un visor interactivo de nubes de palabras con las representaciones de cada tópico.

In [14]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from umap import UMAP

# ==============================================================================
# PROYECCIÓN 2D DEL ESPACIO SEMÁNTICO (TOP 1 REPRESENTATIVO CON OPACIDAD 0.65)
# ==============================================================================

OCULTAR_TOPICO_100 = False


reducer = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42)
coords_2d = reducer.fit_transform(embeddings_naturales)

topic_info = topic_model.get_topic_info()
topic_names = {row['Topic']: row['Name'] for _, row in topic_info.iterrows()}

# 1. Limpieza de taglines para hover
clean_taglines = (
    df['tagline']
    .fillna("")
    .astype(str)
    .apply(lambda x: (x[:80] + "...") if len(x) > 83 else x)
    .replace("", "Sin bajada")
    .values
)

col_id = df["ID"].values if "ID" in df.columns else df["doc_id"].values

# 2. DataFrame base
plot_df = pd.DataFrame({
    'x': coords_2d[:, 0], 
    'y': coords_2d[:, 1],
    'id': col_id,
    'topic': df['topic'].values.astype(int),
    'title': df['title_text'].values,
    'tagline': clean_taglines
})

if OCULTAR_TOPICO_100:
# Excluir outliers (Tópico 100)
    plot_df = plot_df[plot_df['topic'] != 100].copy()

# Mapeo y orden numérico estricto
plot_df['topic_name'] = plot_df['topic'].map(topic_names)
plot_df['topic_label'] = plot_df['topic'].astype(str) + ": " + plot_df['topic_name']

topics_ordenados = sorted(plot_df['topic'].unique())
categorias_ordenadas = [f"{t}: {topic_names[t]}" for t in topics_ordenados]

plot_df['topic_label'] = pd.Categorical(plot_df['topic_label'], categories=categorias_ordenadas, ordered=True)
plot_df = plot_df.sort_values('topic_label')

# 3. Filtrar ÚNICAMENTE el artículo Rank 1 (el más representativo)
top1_df = df_representativos[df_representativos["Rank"] == 1]
dict_reps = top1_df.set_index("ID")[["Similitud_Centroide"]].to_dict(orient="index")

plot_df["es_rep"] = plot_df["id"].isin(dict_reps)
plot_df["rep_sim"] = plot_df["id"].map(lambda x: dict_reps[x]["Similitud_Centroide"] if x in dict_reps else None)

# 4. Renderizado Plotly
fig = px.scatter(
    plot_df, 
    x='x', 
    y='y', 
    color='topic_label', 
    category_orders={"topic_label": categorias_ordenadas},
    custom_data=['topic_label', 'title', 'tagline']
)

fig.update_traces(
    hovertemplate=(
        "<b>Tópico:</b> %{customdata[0]}<br>"
        "<b>Título:</b> %{customdata[1]}<br>"
        "<i>%{customdata[2]}</i><extra></extra>"
    ), 
    marker=dict(size=5, opacity=0.65)
)

# Capa de estrellas doradas (solo el Top 1 por tópico)
reps_df = plot_df[plot_df["es_rep"]].copy()

fig.add_trace(
    go.Scatter(
        x=reps_df['x'],
        y=reps_df['y'],
        mode='markers',
        name='⭐ Más representativo',
        marker=dict(
            symbol='star',
            size=12,
            color='#FFD700',
            line=dict(width=1.2, color='#111111')
        ),
        customdata=reps_df[['topic_label', 'title', 'tagline', 'rep_sim']],
        hovertemplate=(
            "⭐ <b>Artículo más representativo</b><br>"
            "<b>Similitud Centroide:</b> %{customdata[3]:.4f}<br>"
            "<b>Tópico:</b> %{customdata[0]}<br>"
            "<b>Título:</b> %{customdata[1]}<br>"
            "<i>%{customdata[2]}</i><extra></extra>"
        )
    )
)

fig.update_layout(
    title="<b>Mapa de Artículos (UMAP 2D)</b>", 
    plot_bgcolor='rgba(0,0,0,0)', 
    paper_bgcolor='#fff', 
    height=750, 
    hovermode="closest",
    xaxis=dict(showticklabels=False, zeroline=False, showgrid=False, title=""),
    yaxis=dict(showticklabels=False, zeroline=False, showgrid=False, title=""),
    legend=dict(title="<b>Tópicos</b>", orientation="v", y=1, x=1.02)
)

fig.write_html(GRAFICOS_DIR / "03_espacio_documentos.html")
fig.show()

In [15]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from umap import UMAP
from sklearn.metrics.pairwise import cosine_similarity

# ==============================================================================
# MAPA SEMÁNTICO DE TÓPICOS (TOPIC LANDSCAPE 2D)
# ==============================================================================

# 1. Obtener información de tópicos excluyendo outliers (100)
info_df = topic_model.get_topic_info()
info_tematicos = info_df[info_df["Topic"] != 100].copy()

# 2. Calcular el centroide semántico global de cada tópico en alta dimensión
topic_ids = info_tematicos["Topic"].tolist()
centroides_hd = []
volumenes = []
nombres_limpios = []
palabras_clave_hover = []

for t_id in topic_ids:
    idx_topico = df[df["topic"] == t_id].index.tolist()
    centroide = embeddings_naturales[idx_topico].mean(axis=0)
    centroides_hd.append(centroide)
    volumenes.append(len(idx_topico))
    
    # Extraer palabras clave para el nombre y el tooltip
    keywords = [w for w, _ in topic_model.get_topic(t_id)]
    nombres_limpios.append(f"{t_id}: " + "_".join(keywords[:3]))
    palabras_clave_hover.append("<br>• " + "<br>• ".join(keywords[:6]))

centroides_hd = np.array(centroides_hd)

# 3. Proyectar los centroides a 2D conservando distancias relativas
reducer_topics = UMAP(
    n_neighbors=min(8, len(topic_ids) - 1),
    n_components=2,
    min_dist=0.3,
    metric="cosine",
    random_state=42
)
coords_topics_2d = reducer_topics.fit_transform(centroides_hd)

# 4. DataFrame para visualización
df_mapa_topicos = pd.DataFrame({
    "Topic": topic_ids,
    "Nombre": nombres_limpios,
    "x": coords_topics_2d[:, 0],
    "y": coords_topics_2d[:, 1],
    "Volumen": volumenes,
    "Keywords": palabras_clave_hover
})

# 5. Renderizado Plotly
fig = go.Figure()

# Capa de burbujas con etiquetas flotantes de texto
fig.add_trace(
    go.Scatter(
        x=df_mapa_topicos["x"],
        y=df_mapa_topicos["y"],
        mode="markers+text",
        text=df_mapa_topicos["Topic"].astype(str),
        textposition="middle center",
        textfont=dict(size=12, color="#ffffff", family="Arial Black"),
        marker=dict(
            size=df_mapa_topicos["Volumen"],
            sizemode="area",
            sizeref=2.0 * max(df_mapa_topicos["Volumen"]) / (65 ** 2),  # escala visual equilibrada
            sizemin=14,
            color=df_mapa_topicos["Topic"],
            colorscale="Turbo",
            line=dict(width=1.5, color="#222222"),
            opacity=0.88
        ),
        customdata=df_mapa_topicos[["Topic", "Nombre", "Volumen", "Keywords"]],
        hovertemplate=(
            "<b>%{customdata[1]}</b><br>"
            "<b>Volumen total:</b> %{customdata[2]} notas<br>"
            "<b>Términos representativos:</b>%{customdata[3]}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title="<b>Mapa Semántico de Tópicos (Topic Landscape)</b>",
    template="plotly_white",
    height=750,
    hovermode="closest",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    showlegend=False
)

fig.write_html(GRAFICOS_DIR / "03_mapa_topicos.html")
fig.show()

In [ ]:
import plotly.graph_objects as go
from wordcloud import WordCloud
import pandas as pd
import numpy as np

# ==============================================================================
# WORDCLOUDS USANDO LA REPRESENTACIÓN OFICIAL Y DIRECTA DE BERTopic
# ==============================================================================

info_df = topic_model.get_topic_info()
topicos = sorted(info_df["Topic"].astype(int).unique())

fig = go.Figure()
dropdown_buttons = []

# Iteramos directo sobre lo que BERTopic guardó para cada tópico
for idx, t_id in enumerate(topicos):
    # Extraemos la lista oficial de pares (palabra, peso)
    if t_id in topic_model.topic_representations_:
        palabras_pesos_raw = topic_model.topic_representations_[t_id]
    else:
        palabras_pesos_raw = topic_model.get_topic(t_id)
        
    if not palabras_pesos_raw:
        continue

    # Diccionario exacto de palabras y pesos del modelo
    freq_dict = {word: score for word, score in palabras_pesos_raw if score > 0}
    
    if not freq_dict:
        continue

    # Nombres y paleta
    vol_notas = len(df[df["topic"] == t_id])
    primeras_3 = list(freq_dict.keys())[:3]
    nombre_label = f"Tópico {t_id}: " + "_".join(primeras_3) + f" ({vol_notas} notas)"
    colormap = "viridis"

    # Generación de la nube con los pesos idénticos a get_topic()
    wc = WordCloud(
        width=850,
        height=450,
        background_color="white",
        colormap=colormap,
        random_state=42
    ).generate_from_frequencies(freq_dict)

    img_array = np.array(wc.to_image())

    es_visible = (idx == 0)

    fig.add_trace(
        go.Image(
            z=img_array,
            name=nombre_label,
            visible=es_visible,
            hoverinfo="skip"
        )
    )

    visibilidad = [False] * len(topicos)
    visibilidad[idx] = True

    dropdown_buttons.append(
        dict(
            label=nombre_label,
            method="update",
            args=[
                {"visible": visibilidad}
            ]
        )
    )

primer_nombre = dropdown_buttons[0]["label"] if dropdown_buttons else ""

fig.update_layout(
    template="plotly_white",
    height=580,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.18,
            yanchor="top",
            bgcolor="#ffffff",
            bordercolor="#cccccc",
            font=dict(size=12)
        )
    ]
)

fig.write_html(GRAFICOS_DIR / "04_wordclouds_topicos.html")
fig.show()

### Guardado del modelo definitivo y dataset depurado

Serializa el modelo final con la numeración ordenada (`modelo_completo`) y exporta `df_completo.parquet` conservando únicamente las columnas necesarias para los módulos de etiquetas (`06`) y recomendación (`07`).

In [ ]:
# GUARDADO

# 1. El modelo BERTopic actualizado (con tópicos renumerados, 100 y 80 keywords)
topic_model.save(
    str(OUTPUT_DIR / "modelo_completo"), 
    serialization="safetensors", 
    save_ctfidf=True,
    save_embedding_model=False 
)

2026-11-04 07:16:36,754 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


In [ ]:
# 1. Definir las columnas estrictamente necesarias
columnas_finales = [
    'doc_id',                    # O 'doc_id' si es tu clave primaria consistente
    'source_url',
    'title_text',
    'tagline',
    'autor_names',
    'post_date',
    'post_date_dt',
    'category',
    'post_tag',
    'topic',                 # Asignación final (0 a 21 + 100)
    'document_text_final'    # Texto lematizado/limpio
]

# 2. Quedarse solo con lo limpio
df_limpio = df[[c for c in columnas_finales if c in df.columns]].copy()

# 3. Guardar el parquet depurado
df_limpio.to_parquet(OUTPUT_DIR / "df_completo.parquet", index=False)

print(f"DataFrame depurado guardado. De {len(df.columns)} columnas quedaron {len(df_limpio.columns)}:")
print(df_limpio.columns.tolist())

DataFrame depurado guardado. De 21 columnas quedaron 11:
['doc_id', 'source_url', 'title_text', 'tagline', 'autor_names', 'post_date', 'post_date_dt', 'category', 'post_tag', 'topic', 'document_text_final']


## Análisis de autores por tópico

Desanida los artículos con más de un autor y calcula volumen de publicación y grado de especialización de cada autor dentro de cada tópico.

In [ ]:
# Desanidar la columna 'autor_names' separando coautorías por coma o punto y coma
df_autores = (
    df.dropna(subset=["autor_names"])
    .copy()
    .assign(autor=lambda d: d["autor_names"].astype(str).str.replace(";", ",").str.split(","))
    .explode("autor")
    .assign(autor=lambda d: d["autor"].str.strip())
    .loc[lambda d: ~d["autor"].str.lower().isin(["", "nan", "none", "sin autor"])]
    .reset_index(drop=True)
)

print(f"Autores únicos: {df_autores['autor'].nunique():,}")

# Diccionario con el total histórico de notas por autor (para medir especialización)
total_notas_por_autor = df_autores.groupby("autor")["doc_id"].nunique().to_dict()

Autores únicos: 1,694


In [ ]:
def autores_dominantes_terminal(topico_id=100, top_n=15):
    # Filtrar notas del tópico
    df_t = df_autores[df_autores["topic"] == topico_id]
    total_notas_topico = df[df["topic"] == topico_id]["doc_id"].nunique()
    total_autores_topico = df_t["autor"].nunique()
    nombre_topico = topic_names.get(topico_id, f"Tópico {topico_id}")

    if df_t.empty:
        print(f"No hay firmas registradas para el Tópico {topico_id}.")
        return

    # Conteo de notas en el tópico
    ranking = (
        df_t.groupby("autor")["doc_id"]
        .nunique()
        .reset_index(name="notas_en_topico")
    )

    # Métricas
    ranking["total_autor"] = ranking["autor"].map(total_notas_por_autor)
    ranking["pct_del_topico"] = (ranking["notas_en_topico"] / total_notas_topico * 100).round(2)
    ranking["pct_especializacion"] = (ranking["notas_en_topico"] / ranking["total_autor"] * 100).round(2)

    ranking = ranking.sort_values(by=["notas_en_topico", "pct_especializacion"], ascending=[False, False]).head(top_n)

    # Imprimir en consola formateado
    print("=" * 85)
    print(f"AUTORES — TÓPICO {topico_id}: {nombre_topico}")
    print(f"Total notas en el tópico: {total_notas_topico} | Total autores únicos: {total_autores_topico}")
    print("=" * 85)
    print(f"{'#':<3} {'Autor':<30} {'En Tópico':<11} {'Total Autor':<13} {'% del Tópico':<14} {'% Autor En Tópico'}")
    print("-" * 85)

    for i, (_, row) in enumerate(ranking.iterrows(), 1):
        print(
            f"{i:<3} {row['autor'][:28]:<30} "
            f"{row['notas_en_topico']:<11} "
            f"{row['total_autor']:<13} "
            f"{row['pct_del_topico']:<14.2f} "
            f"{row['pct_especializacion']:.2f}%"
        )
    print("=" * 85)

In [ ]:
# Ejecutar para Outliers (100) o cualquier otro tópico
autores_dominantes_terminal(topico_id=1, top_n=30)

AUTORES — TÓPICO 1: 1_literario_autor_escritor_poeta
Total notas en el tópico: 392 | Total autores únicos: 274
#   Autor                          En Tópico   Total Autor   % del Tópico   % Autor En Tópico
-------------------------------------------------------------------------------------
1   Julieta Greco                  13          30            3.32           43.33%
2   María Moreno                   9           13            2.30           69.23%
3   Cristian Alarcón               7           25            1.79           28.00%
4   Sonia Budassi                  7           29            1.79           24.14%
5   Julián Gorodischer             5           12            1.28           41.67%
6   Alejandra Laera                4           4             1.02           100.00%
7   Cora Gamarnik                  4           4             1.02           100.00%
8   Mariana Enríquez               4           9             1.02           44.44%
9   Hinde Pomeraniec               4       